In [0]:
 from pyspark.sql import SparkSession
 from pyspark.sql.functions import *

 spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

 data = [
     (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
  (7, "C007", "Mobile", "18000", "2024-01-07"),
  (8, "C008", "Watch", "8000", "2024-01-07")
]

In [0]:
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)

In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- updated_date: string (nullable = true)



In [0]:
df=df.withColumn("amount",col("amount").cast("int"))
df=df.fillna({
    'amount':0
})

In [0]:
### TASK LIST
### Task 1: Handle NULLs
### Replace NULL amount with 1000
### Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date
### Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low
### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High

In [0]:
df=df.withColumn("updated_date",col('updated_date').cast("date"))


In [0]:
df.printSchema()


root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = false)
 |-- updated_date: date (nullable = true)
 |-- bonus: double (nullable = true)



In [0]:
df=df.withColumn("bonus",col('amount')*0.1)
df.show()

+--------+-----------+----------+------+------------+------+
|order_id|customer_id|   product|amount|updated_date| bonus|
+--------+-----------+----------+------+------------+------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|
|       2|       C002|    Mobile|     0|  2024-01-02|   0.0|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|
|       5|       C005|Headphones|     0|  2024-01-05|   0.0|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|
+--------+-----------+----------+------+------------+------+



In [0]:
df=df.withColumn("category",when(col('amount')>=20000,'High').otherwise("low"))
df.show()

+--------+-----------+----------+------+------------+------+--------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|
+--------+-----------+----------+------+------------+------+--------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|
|       2|       C002|    Mobile|     0|  2024-01-02|   0.0|     low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|    High|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|
|       5|       C005|Headphones|     0|  2024-01-05|   0.0|     low|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     low|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     low|
+--------+-----------+----------+------+------------+------+--------+



In [0]:
def grade(amount):
    if amount<10000:
        return "low"
    elif amount >= 10000 and amount<= 30000:
        return "medium"
    
    else:
        return "high"   

In [0]:
grade_udf=udf(grade,StringType())
df=df.withColumn("final_grade_udf",grade_udf(col('amount')))
df.show()

+--------+-----------+----------+------+------------+------+--------+-----------+---------------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|final_grade|final_grade_udf|
+--------+-----------+----------+------+------------+------+--------+-----------+---------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|       high|           high|
|       2|       C002|    Mobile|     0|  2024-01-02|   0.0|     low|        low|            low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|    High|     medium|         medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|       high|           high|
|       5|       C005|Headphones|     0|  2024-01-05|   0.0|     low|        low|            low|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|     medium|         medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     low|     medium|         medium|
|       8|       C00

In [0]:
df.write \
    .mode("overwrite") \
    .parquet("/Volumes/volume1/phase5/subbu")

In [0]:
existing_df = spark.read.parquet("/Volumes/volume1/phase5/subbu")
existing_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,final_grade,final_grade_udf
3,C003,Tablet,20000,2024-01-03,2000.0,High,medium,medium
6,C006,Camera,30000,2024-01-06,3000.0,High,medium,medium
7,C007,Mobile,18000,2024-01-07,1800.0,low,medium,medium
5,C005,Headphones,0,2024-01-05,0.0,low,low,low
1,C001,Laptop,50000,2024-01-01,5000.0,High,high,high
4,C004,Laptop,55000,2024-01-04,5500.0,High,high,high
2,C002,Mobile,0,2024-01-02,0.0,low,low,low
8,C008,Watch,8000,2024-01-07,800.0,low,low,low


In [0]:
last_entry_date=existing_df.select(max(col('updated_date'))).collect()[0][0]
print(last_entry_date)

2024-01-07


INCREMENTAL LOAD

In [0]:
data=[(3, "C003", "Tablet", "22000", "2024-01-08", 2200,"High", "high", "high")]

In [0]:
columns = ["order_id", "customer_id", "product", "amount", "updated_date","bonus","category","final_grade","final_grade_udf"]
df1 = spark.createDataFrame(data, columns)

In [0]:
df1.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- updated_date: string (nullable = true)
 |-- bonus: long (nullable = true)
 |-- category: string (nullable = true)
 |-- final_grade: string (nullable = true)
 |-- final_grade_udf: string (nullable = true)



In [0]:
df1=df1.withColumn("updated_date",col('updated_date').cast("date"))
df1=df1.withColumn("bonus",col('bonus').cast("int"))


In [0]:
incremental_df = df1.filter(
    col("updated_date") > last_entry_date
)
incremental_df.show()

+--------+-----------+-------+------+------------+-----+--------+-----------+---------------+
|order_id|customer_id|product|amount|updated_date|bonus|category|final_grade|final_grade_udf|
+--------+-----------+-------+------+------------+-----+--------+-----------+---------------+
|       3|       C003| Tablet| 22000|  2024-01-08| 2200|    High|       high|           high|
+--------+-----------+-------+------+------------+-----+--------+-----------+---------------+



In [0]:
filtered_existing = existing_df.join(
    incremental_df,
    "order_id",
    "left_anti"
)

In [0]:
final_df = filtered_existing.union(incremental_df)
final_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,final_grade,final_grade_udf
6,C006,Camera,30000,2024-01-06,3000.0,High,medium,medium
7,C007,Mobile,18000,2024-01-07,1800.0,low,medium,medium
5,C005,Headphones,0,2024-01-05,0.0,low,low,low
1,C001,Laptop,50000,2024-01-01,5000.0,High,high,high
4,C004,Laptop,55000,2024-01-04,5500.0,High,high,high
2,C002,Mobile,0,2024-01-02,0.0,low,low,low
8,C008,Watch,8000,2024-01-07,800.0,low,low,low
3,C003,Tablet,22000,2024-01-08,2200.0,High,high,high
